# 02 — Training Pipeline

**Project:** UREP 32-0210-250078 | Crack Classification

**Classes:** Debonding | Flexural | Shear | Multi-Crack | No Crack | Others

3-stage transfer learning with InceptionV3:
1. **Stage 1** — Feature extraction (frozen backbone, LR=1e-3)
2. **Stage 2** — Partial fine-tuning (unfreeze from mixed7, LR=1e-4)
3. **Stage 3** — Full fine-tuning (all layers, LR=1e-5)

**Metrics tracked per epoch** (logged to CSV + TensorBoard):
- `loss` / `val_loss` — categorical cross-entropy
- `accuracy` / `val_accuracy`
- `precision` / `val_precision`
- `recall` / `val_recall`

**Class balancing:** `class_weight` scales loss by inverse class frequency — minority classes (flexural, multi_crack, others) get ~2.7–2.9x loss weight vs. majority (no_crack) at ~0.36x.

Per-stage batch sizes optimized for available GPU VRAM to prevent OOM errors.

## Setup

In [1]:
import sys
sys.path.insert(0, "..")

import os
import tensorflow as tf
import numpy as np

import config
from src.dataset import prepare_dataset, split_dataset, get_generators, compute_class_weights
from src.model import (
    build_model, freeze_backbone, unfreeze_from, unfreeze_all,
    compile_model, get_callbacks,
)
from src.evaluation import plot_training_history
from src.device import print_device_summary

# Detect hardware and get optimal training config
device_config = print_device_summary()

# Per-stage batch sizes (auto-detected or from config defaults)
STAGE_BATCH = device_config.get("batch_sizes", {
    1: config.STAGE1_BATCH_SIZE,
    2: config.STAGE2_BATCH_SIZE,
    3: config.STAGE3_BATCH_SIZE,
})

print(f"\nTensorFlow version: {tf.__version__}")
print(f"NUM_CLASSES: {config.NUM_CLASSES}")
print(f"CLASS_NAMES: {config.CLASS_NAMES}")

  PyTorch may see the GPU, but TF needs matching CUDA/cuDNN.
  Diagnostics:
    1. Verify CUDA 12.x is installed (not just the NVIDIA driver)
    2. Verify cuDNN 9.x is installed and on PATH
    3. Run: python -c "import tensorflow as tf; print(tf.sysconfig.get_build_info())"
  Training will proceed on CPU (significantly slower).

DEVICE SUMMARY
  GPU:         Not detected (CPU-only mode)
  RAM:         13.9 GB
  CPU cores:   16 logical
  Platform:    Windows

RECOMMENDED TRAINING CONFIG
----------------------------------------
  Stage 1 batch size:  8
  Stage 2 batch size:  8
  Stage 3 batch size:  4
  Workers:              2
  Multiprocessing:      False


TensorFlow version: 2.20.0
NUM_CLASSES: 6
CLASS_NAMES: ['debonding', 'flexural', 'shear', 'multi_crack', 'no_crack', 'others']


## Data Preparation

First, map the QU nested dataset to flat class folders, then split into train/val/test.

In [2]:
# Step 1: Prepare flat class folders from QU dataset (only run once)
print("Preparing dataset from QU structure...")
prepare_dataset()

# Step 2: Split into train/val/test (only run once — skip if already split)
if not os.path.exists(config.SPLIT_DIR):
    stats = split_dataset()
else:
    print(f"\nSplit directory already exists: {config.SPLIT_DIR}")
    print("Delete it and re-run this cell to re-split.")

Preparing dataset from QU structure...
  debonding   :   5577 images (already exists, skipping)
  flexural    :   2558 images (already exists, skipping)
  shear       :   9587 images (already exists, skipping)
  multi_crack :   2455 images (already exists, skipping)
  no_crack    :  20000 images (already exists, skipping)
  others      :   2610 images (already exists, skipping)

Dataset preparation complete: 42,787 images across 6 classes
  debonding   :   5577
  flexural    :   2558
  shear       :   9587
  multi_crack :   2455
  no_crack    :  20000
  others      :   2610

Split directory already exists: d:\urep\data\split
Delete it and re-run this cell to re-split.


In [ ]:
# Create data generators for Stage 1
train_gen, val_gen, test_gen = get_generators(batch_size=STAGE_BATCH[1])

# Print per-class file counts for each split
print(f"\n{'='*70}")
print("DATASET FILE COUNTS PER CLASS")
print(f"{'='*70}")
print(f"{'Class':<14} {'Train':>8} {'Val':>8} {'Test':>8} {'Total':>8}")
print("-" * 70)
total_train = total_val = total_test = 0
for cls_name in config.CLASS_NAMES:
    tr = len(os.listdir(os.path.join(config.SPLIT_DIR, "train", cls_name)))
    va = len(os.listdir(os.path.join(config.SPLIT_DIR, "val", cls_name)))
    te = len(os.listdir(os.path.join(config.SPLIT_DIR, "test", cls_name)))
    total_train += tr; total_val += va; total_test += te
    print(f"  {cls_name:<12} {tr:>8,} {va:>8,} {te:>8,} {tr+va+te:>8,}")
print("-" * 70)
print(f"  {'TOTAL':<12} {total_train:>8,} {total_val:>8,} {total_test:>8,} {total_train+total_val+total_test:>8,}")
print(f"{'='*70}")

print(f"\nStage 1 batch size: {STAGE_BATCH[1]}")
print(f"Training batches:   {len(train_gen)}")
print(f"Validation batches: {len(val_gen)}")
print(f"Test batches:       {len(test_gen)}")

In [ ]:
# Compute class weights for imbalance handling
class_weights = compute_class_weights()

def print_stage_summary(stage, batch_size, epochs, lr, train_gen, class_weights):
    """Print training summary before each stage starts."""
    n_train = train_gen.samples
    n_val = val_gen.samples
    n_batches = len(train_gen)
    total_images_seen = n_train * epochs  # images seen across all epochs (with augmentation)

    print(f"\n{'='*65}")
    print(f"STAGE {stage} TRAINING PLAN")
    print(f"{'='*65}")
    print(f"  Training files:     {n_train:>8,}  (augmented on-the-fly each epoch)")
    print(f"  Validation files:   {n_val:>8,}  (no augmentation)")
    print(f"  Batch size:         {batch_size:>8}")
    print(f"  Batches/epoch:      {n_batches:>8,}")
    print(f"  Max epochs:         {epochs:>8}")
    print(f"  Learning rate:      {lr:>12.1e}")
    print(f"  Total images seen:  {total_images_seen:>8,}  ({epochs} epochs x {n_train:,} files)")
    print(f"{'-'*65}")
    print(f"  {'Class':<14} {'Files':>7} {'Weight':>8} {'Effective':>10}")
    print(f"  {'-'*50}")
    for idx, cls_name in enumerate(config.CLASS_NAMES):
        cls_dir = os.path.join(config.SPLIT_DIR, "train", cls_name)
        count = len(os.listdir(cls_dir))
        w = class_weights[idx]
        print(f"  {cls_name:<14} {count:>7,} {w:>8.3f} {count * w:>10,.1f}")
    print(f"{'='*65}")

## Build Model

In [5]:
model, base_model = build_model()
print(f"Total parameters: {model.count_params():,}")
model.summary(show_trainable=True)

Total parameters: 22,362,022


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)      ┃ Output Shape    ┃   Param # ┃ Connected to   ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ input_layer       │ (None, 299,     │         0 │ -              │   -   │
│ (InputLayer)      │ 299, 3)         │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ conv2d (Conv2D)   │ (None, 149,     │       864 │ input_layer[0… │   Y   │
│                   │ 149, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ batch_normalizat… │ (None, 149,     │        96 │ conv2d[0][0]   │   Y   │
│ (BatchNormalizat… │ 149, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ activation        │ (None, 149,     │         0 │ batch_normali… │   -   │
│ (Activation)      │ 149, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ conv2d_1 (Conv2D) │ (None, 147,     │     9,216 │ activation[0]… │   Y   │
│                   │ 147, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ batch_normalizat… │ (None, 147,     │        96 │ conv2d_1[0][0] │   Y   │
│ (BatchNormalizat… │ 147, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ activation_1      │ (None, 147,     │         0 │ batch_normali… │   -   │
│ (Activation)      │ 147, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ conv2d_2 (Conv2D) │ (None, 147,     │    18,432 │ activation_1[… │   Y   │
│                   │ 147, 64)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ batch_normalizat… │ (None, 147,     │       192 │ conv2d_2[0][0] │   Y   │
│ (BatchNormalizat… │ 147, 64)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ activation_2      │ (None, 147,     │         0 │ batch_normali… │   -   │
│ (Activation)      │ 147, 64)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ max_pooling2d     │ (None, 73, 73,  │         0 │ activation_2[… │   -   │
│ (MaxPooling2D)    │ 64)             │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ conv2d_3 (Conv2D) │ (None, 73, 73,  │     5,120 │ max_pooling2d… │   Y   │
│                   │ 80)             │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ batch_normalizat… │ (None, 73, 73,  │       240 │ conv2d_3[0][0] │   Y   │
│ (BatchNormalizat… │ 80)             │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ activation_3      │ (None, 73, 73,  │         0 │ batch_normali… │   -   │
│ (Activation)      │ 80)             │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ conv2d_4 (Conv2D) │ (None, 71, 71,  │   138,240 │ activation_3[… │   Y   │
│                   │ 192)            │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ batch_normalizat… │ (None, 71, 71,  │       576 │ conv2d_4[0][0] │   Y   │
│ (BatchNormalizat… │ 192)            │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ activation_4      │ (None, 71, 71,  │         0 │ batch_normali… │   - 

 Total params: 22,362,022 (85.30 MB)

 Trainable params: 22,327,078 (85.17 MB)

 Non-trainable params: 34,944 (136.50 KB)

## Stage 1 — Feature Extraction

Frozen InceptionV3 backbone, train only the classification head.

In [ ]:
freeze_backbone(base_model)
compile_model(model, learning_rate=config.STAGE1_LR, stage=1)
print_stage_summary(1, STAGE_BATCH[1], config.STAGE1_EPOCHS, config.STAGE1_LR, train_gen, class_weights)

history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=config.STAGE1_EPOCHS,
    class_weight=class_weights,
    callbacks=get_callbacks(stage=1),
)

# Print all tracked metrics for Stage 1
print(f"\n{'='*50}")
print(f"STAGE 1 RESULTS — Feature Extraction")
print(f"{'='*50}")
print(f"  Best val_accuracy:  {max(history1.history['val_accuracy']):.4f}")
print(f"  Best val_loss:      {min(history1.history['val_loss']):.4f}")
print(f"  Best val_precision: {max(history1.history['val_precision']):.4f}")
print(f"  Best val_recall:    {max(history1.history['val_recall']):.4f}")
print(f"  Final train_loss:   {history1.history['loss'][-1]:.4f}")
print(f"  Final train_acc:    {history1.history['accuracy'][-1]:.4f}")
print(f"  Epochs completed:   {len(history1.history['loss'])}/{config.STAGE1_EPOCHS}")
print(f"  CSV log saved to:   outputs/logs/stage1_metrics.csv")
print(f"{'='*50}")

## Stage 2 — Partial Fine-Tuning

Unfreeze from `mixed7` onwards, lower learning rate. Recreate generators with Stage 2 batch size.

In [ ]:
# Recreate generators with Stage 2 batch size
if STAGE_BATCH[2] != STAGE_BATCH[1]:
    train_gen, val_gen, _ = get_generators(batch_size=STAGE_BATCH[2])
    print(f"Recreated generators with batch_size={STAGE_BATCH[2]}")

unfreeze_from(base_model, layer_name="mixed7")
compile_model(model, learning_rate=config.STAGE2_LR, stage=2)
print_stage_summary(2, STAGE_BATCH[2], config.STAGE2_EPOCHS, config.STAGE2_LR, train_gen, class_weights)

history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=config.STAGE2_EPOCHS,
    class_weight=class_weights,
    callbacks=get_callbacks(stage=2),
)

# Print all tracked metrics for Stage 2
print(f"\n{'='*50}")
print(f"STAGE 2 RESULTS — Partial Fine-Tuning (from mixed7)")
print(f"{'='*50}")
print(f"  Best val_accuracy:  {max(history2.history['val_accuracy']):.4f}")
print(f"  Best val_loss:      {min(history2.history['val_loss']):.4f}")
print(f"  Best val_precision: {max(history2.history['val_precision']):.4f}")
print(f"  Best val_recall:    {max(history2.history['val_recall']):.4f}")
print(f"  Final train_loss:   {history2.history['loss'][-1]:.4f}")
print(f"  Final train_acc:    {history2.history['accuracy'][-1]:.4f}")
print(f"  Epochs completed:   {len(history2.history['loss'])}/{config.STAGE2_EPOCHS}")
print(f"  CSV log saved to:   outputs/logs/stage2_metrics.csv")
print(f"{'='*50}")

## Stage 3 — Full Fine-Tuning

All layers trainable, very low learning rate. Recreate generators with smaller batch size to avoid OOM.

In [ ]:
# Recreate generators with Stage 3 batch size (smaller to prevent OOM)
if STAGE_BATCH[3] != STAGE_BATCH[2]:
    train_gen, val_gen, _ = get_generators(batch_size=STAGE_BATCH[3])
    print(f"Recreated generators with batch_size={STAGE_BATCH[3]}")

unfreeze_all(base_model)
compile_model(model, learning_rate=config.STAGE3_LR, stage=3)
print_stage_summary(3, STAGE_BATCH[3], config.STAGE3_EPOCHS, config.STAGE3_LR, train_gen, class_weights)

history3 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=config.STAGE3_EPOCHS,
    class_weight=class_weights,
    callbacks=get_callbacks(stage=3),
)

# Print all tracked metrics for Stage 3
print(f"\n{'='*50}")
print(f"STAGE 3 RESULTS — Full Fine-Tuning (all layers)")
print(f"{'='*50}")
print(f"  Best val_accuracy:  {max(history3.history['val_accuracy']):.4f}")
print(f"  Best val_loss:      {min(history3.history['val_loss']):.4f}")
print(f"  Best val_precision: {max(history3.history['val_precision']):.4f}")
print(f"  Best val_recall:    {max(history3.history['val_recall']):.4f}")
print(f"  Final train_loss:   {history3.history['loss'][-1]:.4f}")
print(f"  Final train_acc:    {history3.history['accuracy'][-1]:.4f}")
print(f"  Epochs completed:   {len(history3.history['loss'])}/{config.STAGE3_EPOCHS}")
print(f"  CSV log saved to:   outputs/logs/stage3_metrics.csv")
print(f"{'='*50}")

## Training Curves

In [ ]:
plot_training_history(
    [history1, history2, history3],
    stage_names=["Feature Extraction", "Partial Fine-Tuning", "Full Fine-Tuning"],
)

## Save Final Model

In [ ]:
os.makedirs(config.MODEL_DIR, exist_ok=True)
final_model_path = os.path.join(config.MODEL_DIR, "best_model.keras")
model.save(final_model_path)
print(f"Model saved to: {final_model_path}")

---

## (Optional) Hyperparameter Tuning with Keras Tuner

**If you skip this section**, the 3-stage training above uses the **default hyperparameters** from `config.py`:
- Dropout: 0.4 | L2 reg: 1e-4 | Dense layers: 256 → 128
- LR: 1e-3 → 1e-4 → 1e-5 (per stage)
- These defaults are reasonable starting points from literature/experimentation.

**If you run this section**, Bayesian optimization searches over:
- `dropout_rate`: 0.2–0.5 | `l2_reg`: 1e-3, 1e-4, 1e-5
- `dense_units_1`: 128, 256, 512 | `dense_units_2`: 64, 128, 256
- `learning_rate`: 1e-3, 5e-4, 1e-4

It runs up to 30 trials (each 10 epochs, Stage 1 only) and reports the best combination. You would then update `config.py` with those values and re-run the full 3-stage pipeline.

In [ ]:
# Uncomment to run hyperparameter tuning

# import keras_tuner as kt
# from src.model import build_tunable_model
#
# tuner = kt.BayesianOptimization(
#     build_tunable_model,
#     objective="val_accuracy",
#     max_trials=30,
#     seed=config.RANDOM_SEED,
#     directory=os.path.join(config.OUTPUT_DIR, "tuner"),
#     project_name="crack_classification",
# )
#
# tuner.search(
#     train_gen,
#     validation_data=val_gen,
#     epochs=10,
#     class_weight=class_weights,
#     callbacks=[tf.keras.callbacks.EarlyStopping("val_loss", patience=3)],
# )
#
# print("\nBest hyperparameters:")
# best_hp = tuner.get_best_hyperparameters(1)[0]
# for param in ["dropout_rate", "l2_reg", "dense_units_1", "dense_units_2", "learning_rate"]:
#     print(f"  {param}: {best_hp.get(param)}")